# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Роман Бойко'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-32'
НОМЕР        = 4       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

with open('lr3_variants_student.json', 'r', encoding='utf-8') as f:
    ВАРІАНТИ = json.load(f)
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 4   ·   Роман Бойко   ·   ШІ-32

Ваші вісім стовпців:
   1. Телефон
   2. Ускладнення
   3. Перебіг
   4. Виписка
   5. Ліжко днів
   6. Д-дим1
   7. Фібрин2
   8. МНВ2

Ваша пара для аналізу зв’язку: ступінь КТ та ЛН

Ваше додаткове питання:
   У чому різниця між шкалою інтервалів і шкалою відношень? Знайдіть у файлі по одному прикладу кожної та поясніть, яка операція допустима для другої, але не для першої.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Ускладнення,Перебіг,Виписка,Ліжко днів,Д-дим1,Фібрин2,МНВ2
0,0958469788 дружина Ганна,"тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС",важкий,2022-03-10 00:00:00,14,NaN,2.30,239
1,"0671778486, 0994887663 дружина Оксана",NaN,важкий,2022-03-16 00:00:00,22,3316,3.40,1.08
2,"0958768045, чоловік Степан 0951784726",NaN,важкий,2022-01-18 00:00:00,20,NaN,3.27,1.18
3,0959819331. дочка Валентина 0960845205,NaN,важкий,2022-01-13 00:00:00,20,NaN,4.00,1.07
4,"0952294866, дочка Дарина 0678360161",NaN,важкий,2022-03-11 00:00:00,27,NaN,3.37,2.18
5,0959922983 сестра Софія,"Респірат дестрес синдром, емфізема легень, СЛН",важкий,2021-12-26 00:00:00,9(5 у реан),8000,3.00,2015-01-01 00:00:00
6,"0998073702, дружина Наталія 0998382226",NaN,важкий,2022-05-11 00:00:00,18,183,3.12,1.09
7,"0637258771, син Сергій 0968413301",NaN,серед важ,2022-05-28 00:00:00,10,3028,2.40,1.09


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Телефон      120 non-null    object 
 1   Ускладнення  61 non-null     str    
 2   Перебіг      121 non-null    str    
 3   Виписка      120 non-null    object 
 4   Ліжко днів   120 non-null    object 
 5   Д-дим1       59 non-null     object 
 6   Фібрин2      101 non-null    float64
 7   МНВ2         76 non-null     object 
dtypes: float64(1), object(5), str(2)
memory usage: 18.2+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Ускладнення
    непорожніх 61 з 121 | унікальних 35 | типи: str×61
    приклади: ["'тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС'", "'Респірат дестрес синдром, емфізема легень, СЛН'", "'Респірат дестрес синдром, емфізема легень, СЛН ТІШ'", "'Гострий респіраторний дистрес синдром,ТЕЛА?Прогресуюча серц-леген недос'"]

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- Ліжко днів
    непорожніх

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),
    'Телефон': ('Номінальна', 'Неможливо порівняти за номером і ПІЬ хто кращий'),
   'Ускладнення': ('Номінальна','Неможливо визначити що гірше'),
   'Перебіг': ('Порядкова','Маємо стани важкості але не у математичному вигляді'),
   'Виписка': ('Інтервальна','Різниця між датами має сенс і можна визначити умовно нульову'),
   'Ліжко днів': ('Абсолютна','Розрахунок цілих неподільних одиниць і присутній нуль'),
   'Д-дим1': ('Відношень','Лабораторний показник, є абсолютний 0 і значення можна ділити одне на одне'),
   'Фібрин2': ('Відношень','Лабораторний показник, є абсолютний 0 і значення можна ділити одне на одне'),
   'МНВ2': ('Відношень','Лабораторний показник, є абсолютний 0 і значення можна ділити одне на одне'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,Номінальна,Неможливо порівняти за номером і ПІЬ хто кращий
1,Ускладнення,Номінальна,Неможливо визначити що гірше
2,Перебіг,Порядкова,Маємо стани важкості але не у математичному вигляді
3,Виписка,Інтервальна,Різниця між датами має сенс і можна визначити умовно нул...
4,Ліжко днів,Абсолютна,Розрахунок цілих неподільних одиниць і присутній нуль
5,Д-дим1,Відношень,"Лабораторний показник, є абсолютний 0 і значення можна д..."
6,Фібрин2,Відношень,"Лабораторний показник, є абсолютний 0 і значення можна д..."
7,МНВ2,Відношень,"Лабораторний показник, є абсолютний 0 і значення можна д..."


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    р['мода'] = з.mode().tolist()
    р['частоти'] = з.value_counts().to_dict()
    if ш in ('ідентифікатор', 'номінальна'):
        return р
    try:
        р['медіана'] = з.median()
        р['квантилі'] = з.quantile([0.25, 0.5, 0.75]).to_dict()
    except TypeError:
        р['медіана'] = 'Потребує числа'
        р['квантилі'] = 'Потребує числа'
    if ш in ('порядкова'):
        return р
    try:
        р['середнє'] = з.mean()
        р['стандартне відхилення'] = з.std()
    except TypeError:
        р['середнє'] = 'Неможливо обчислити для дати або тексту'
        р['стандартне відхилення'] = 'Неможливо обчислити'
        
    if ш in ('інтервальна'):
        return р
    try:
        р['середнє'] = з.mean()
        р['коефіцієнт варіації'] = (з.std()/р['середнє']) if р['середнє']!=0 else None
    except TypeError:
        р['коефіцієнт варіації'] = 'Неможливо обчислити'
    
    if ш in ('відношень', 'абсолютна'):
        return р
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (Номінальна)
    {'непорожніх': 120, 'унікальних': 120, 'мода': [501320598, 503516199, 504118955, 506188479, 634193039, 635060454, 637056709, 675240137, 677611387, 680156464, 680591483, 681960278, 686795965, 688280258, 955939761, 961280845, 964398286, 967464397, 968181378, 969042163, 990370382, 991566979, 992041263, '0503817805, друж Вікторія 0997803704', '0504887781 Марина', '0505529301 сестра Лариса', '0634104751 внучкаАльона ', '0635244106 син Микола', '0637212835, дружт 0956990854', '0637258771, син Сергій 0968413301', '0637613414 дочка Софія', '0638296017невістка', '0638440932, дружина Ганна 0634845263', '0670383004 друж Марія', '0671076177 дочка Дарина', '0671778486, 0994887663 дружина Оксана', '0672022697, невістка Тетяна 0960438581', '0673257192 подруга', '0675795111, дочка 0954432568', '0675996775, син Сергій 0500314503', '06762394249 брат Роман', '0679110285 подруга Зоряна', '072761514 дочка Катерина', '0951331159, друж Марина 0992123636', '0952294866, дочка Дари

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ускладнення,61,35,Гострий респіраторний дистрес синдром. Гостра серц-леген...,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Д-дим1,59.0,48.0,10000.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Фібрин2,101.0,NaN,NaN,NaN,4.220396,1.884609,0.99,2.7,3.92,5.05,11.1
МНВ2,76.0,51.0,1.24,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
БЕЗГЛУЗДІ = [
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),
    ('Телефон', 'freq', 1, 'Частота появи номеру людини завжди буде 1.'),
    ('Д-дим1', 'top', 10000.0, 'Шукати моду для лабораторного показника сенсу немає.'),
    ('МНВ2', 'freq', 4.0, 'Для частоти потрібне середнє відхилення, а порахувано через наявність дати.')
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,freq,1.0,Частота появи номеру людини завжди буде 1.
1,Д-дим1,top,10000.0,Шукати моду для лабораторного показника сенсу немає.
2,МНВ2,freq,4.0,"Для частоти потрібне середнє відхилення, а порахувано че..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: ступінь КТ × ЛН

ступінь КТ: шкала — не у вашому списку
ЛН: шкала — не у вашому списку

  ступінь КТ      ЛН
0    сердній  ІІІ-IV
1   середній  ІІ-ІІІ
2  критичний      ІІ
3  критичний  ІІІ-IV
4        NaN  ІІ-ІІІ
5   середній  ІІ-ІІІ


In [12]:
from scipy import stats
ОБРАНА_МІРА = 'рангова кореляція Спірмена'        # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = 'ступінь КТ та ЛН вимірюються в порядковій шкалі. Для поєднання "порядкова × порядкова" використовується саме рангова кореляція Спірмена.'     # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# ваш код: обчисліть обрану міру
KT = {
    'легкий': 1, 
    'сердній': 2,  
    'середній': 2, 
    'важкий': 3, 
    'критичний': 4
}
LN = {
    'І': 1, 
    'І-ІІ': 1.5, 
    'ІІ': 2, 
    'ІІ-ІІІ': 2.5, 
    'ІІІ': 3, 
    'ІІІ-IV': 3.5, 
    'IV': 4
}
pair = df[[A, B]].dropna().copy()
pair[A] = pair[A].astype(str).str.strip().map(KT)
pair[B] = pair[B].astype(str).str.strip().map(LN)
pair = pair.dropna()
correlation, p_value = stats.spearmanr(pair[A], pair[B])
print(f"Коефіцієнт Спірмена: {correlation:.4f}")

Коефіцієнт Спірмена: 0.5738


In [13]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
correlation2, p_value2 = stats.pearsonr(pair[A], pair[B])
print(f"Коефіцієнт Пірсона: {correlation2:.4f}")

print("Коефіцієнт Пірсона вимагає шкали відношень та лінійної залежності."
      "Він вважає, що різниця у важкості між ІІ та ІІІ ступенем абсолютно така ж,"
      "як між І та ІІ. Для медичних порядкових шкал це хибне припущення.")

Коефіцієнт Пірсона: 0.5530
Коефіцієнт Пірсона вимагає шкали відношень та лінійної залежності.Він вважає, що різниця у важкості між ІІ та ІІІ ступенем абсолютно така ж,як між І та ІІ. Для медичних порядкових шкал це хибне припущення.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    # 'стовпець': 'спосіб і коротке обґрунтування',
    'Телефон': 'Номінальна (ідентифікатор): не несе цінності, видаляється з датасету.',
    'Ускладнення': 'Номінальна: застосовується One-Hot Encoding.',
    'Перебіг': 'Порядкова: кодування цілими числами зі збереженням важкості.',
    'Виписка': 'Інтервальна: перетворення дати на тривалість.',
    'Ліжко днів': 'Абсолютна: стандартизація або нормалізація.',
    'Д-дим1': 'Відношень: очищення від текстового сміття, заповнення пропусків та стандартизація.',
    'Фібрин2': 'Відношень: заповнення пропусків та стандартизація.',
    'МНВ2': 'Відношень: очищення від сміття, заповнення пропусків та стандартизація.'
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,Номінальна,"Номінальна (ідентифікатор): не несе цінності, видаляєтьс..."
1,Ускладнення,Номінальна,Номінальна: застосовується One-Hot Encoding.
2,Перебіг,Порядкова,Порядкова: кодування цілими числами зі збереженням важко...
3,Виписка,Інтервальна,Інтервальна: перетворення дати на тривалість.
4,Ліжко днів,Абсолютна,Абсолютна: стандартизація або нормалізація.
5,Д-дим1,Відношень,"Відношень: очищення від текстового сміття, заповнення пр..."
6,Фібрин2,Відношень,Відношень: заповнення пропусків та стандартизація.
7,МНВ2,Відношень,"Відношень: очищення від сміття, заповнення пропусків та ..."


In [15]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
# 1 Номінальна ('Ускладнення')
ускладнення_ohe = pd.get_dummies(df['Ускладнення'].astype(str).str.strip(), prefix = "Ускл")
ускладнення_ohe = ускладнення_ohe.iloc[:, :4]

# 2 Відношень ('Фібрин2')
fibr_mean = df['Фібрин2'].mean()
fibr_std = df['Фібрин2'].std()
df['Фібрин2_z_score'] = (df['Фібрин2'] - fibr_mean) / fibr_std

# 3 Порядкова ('Перебіг')
perebig_map = {
    'серед важ': 1, 
    'важкий': 2, 
    'вкрай важкий': 3
}
df['Перебіг_num'] = df['Перебіг'].astype(str).str.strip().map(perebig_map)

result = pd.concat([
    df[['Фібрин2', 'Фібрин2_z_score','Перебіг', 'Перебіг_num']], 
    ускладнення_ohe,
], axis=1)

result.head()

,Фібрин2,Фібрин2_z_score,Перебіг,Перебіг_num,Ускл_ГРДС,Ускл_ГРДС .ТЕЛА,Ускл_ГРДС .ТЕЛА.Поліорганна недостатність,Ускл_ГРДС. ТЕЛА.Набряк легень
0,2.30,-1.018989,важкий,2,False,False,False,False
1,3.40,-0.435314,важкий,2,False,False,False,False
2,3.27,-0.504293,важкий,2,False,False,False,False
3,4.00,-0.116945,важкий,2,False,False,False,False
4,3.37,-0.451232,важкий,2,False,False,False,False


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
    (1, 'Результат лікування', 
     'Середнє арифметичне для категорій "помер" і "жив" не має математичного сенсу.', 
     'Треба рахувати моду або відсоток тих, хто вижив, і тих, хто помер через value_counts().'),
    
    (2, 'Дублювання категорій.', 
     'У висновку сказано про 4 статі, хоча насправді їх 2. Це сталося через відсутність стандартизації тексту ("ж" і "жін", "ч" і "чол").', 
     'Спочатку потрібно очистити дані: замінити "жін" на "ж", а "чол" на "ч", і лише потім рахувати частоти.'),
    
    (3, 'Брудні дані (Ступінь КТ).', 
     '1. для порядкової шкали не можна рахувати середнє, бо відстані між ступенями нерівні. 2. через помилки в словах порядок збився і функція просто посортувала дані за алфавітом.', 
     'Очистити текст, закодувати словником зі збереженням ієрархії, і порахувати медіану.'),
    
    (4, 'Втрата даних при перетворенні типів (КТ у %).', 
     'Команда to_numeric(errors="coerce") перетворила на NaN усі записи з діапазонами, залишивши лише поодинокі числа, що сильно викривило середнє.', 
     'Потрібно написати функцію, яка витягує числа з діапазону і бере їхнє середнє.'),
    
    (5, 'Помилка розмірності.', 
     'Сатурація 0.9% означає смерть. Хтось ввів у відсотках. Це спотворило мінімум і, як наслідок, середнє.', 
     'Усі значення, які менші за за 10, потрібно помножити на 100 перед аналізом.'),
    
    (6, 'Інтервальна шкала (Дати поступлення).', 
     'Коефіцієнт варіації можна рахувати лише для шкал відношень. Дата — це інтервальна шкала без абсолютного нуля, тому вона створює ілюзію стабільності.', 
     'Для дат коефіцієнт варіації рахувати не можна. Можна оцінити стандартне відхилення.'),
    
    (7, 'Порушення умов кореляції Пірсона.', 
     'Пірсон вимагає двох шкал відношень з лінійною залежністю. Застосовувати його до категорій стать і смерть не можна.', 
     'Для двох номінальних ознак потрібно використовувати коефіцієнт Крамера.'),
    
    (8, 'Ділення на нуль.', 
     'У стовпці ФЕРИТ2 і інших бракує даних або всі значення однакові. Знаменник стає нулем, що генерує NaN.', 
     'Перед нормалізацією перевірити стовпці на варіативність і виключити ті, де min == max.'),
    
    (9, 'Хибне видалення пропусків.', 
     'Оскільки в медичних даних багато пропусків, dropna() видалив усі рядки, бо в кожному пацієнті бракувало хоч однієї клітинки.', 
     'Спочатку видалити стовпці, де є хоча б половина пропусків, а в решті — заповнити пропуски медіаною.'),
    
    (10, 'Витік даних.', 
     'Дата розтину відбувається лише тоді, коли пацієнт помер. Використовувати її як предиктор — це намагатися "передбачити" подію за наслідком, який настав після неї.', 
     'Перед навчанням моделі з датасету треба видалити всі змінні, які з\'являються після того, як стався цільовий результат.')
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Результат лікування,"Середнє арифметичне для категорій ""помер"" і ""жив"" не має...","Треба рахувати моду або відсоток тих, хто вижив, і тих, ..."
1,2,Дублювання категорій.,"У висновку сказано про 4 статі, хоча насправді їх 2. Це ...","Спочатку потрібно очистити дані: замінити ""жін"" на ""ж"", ..."
2,3,Брудні дані (Ступінь КТ).,"1. для порядкової шкали не можна рахувати середнє, бо ві...","Очистити текст, закодувати словником зі збереженням ієра..."
3,4,Втрата даних при перетворенні типів (КТ у %).,"Команда to_numeric(errors=""coerce"") перетворила на NaN у...","Потрібно написати функцію, яка витягує числа з діапазону..."
4,5,Помилка розмірності.,Сатурація 0.9% означає смерть. Хтось ввів у відсотках. Ц...,"Усі значення, які менші за за 10, потрібно помножити на ..."
5,6,Інтервальна шкала (Дати поступлення).,Коефіцієнт варіації можна рахувати лише для шкал відноше...,Для дат коефіцієнт варіації рахувати не можна. Можна оці...
6,7,Порушення умов кореляції Пірсона.,Пірсон вимагає двох шкал відношень з лінійною залежністю...,Для двох номінальних ознак потрібно використовувати коеф...
7,8,Ділення на нуль.,У стовпці ФЕРИТ2 і інших бракує даних або всі значення о...,Перед нормалізацією перевірити стовпці на варіативність ...
8,9,Хибне видалення пропусків.,"Оскільки в медичних даних багато пропусків, dropna() вид...","Спочатку видалити стовпці, де є хоча б половина пропускі..."
9,10,Витік даних.,"Дата розтину відбувається лише тоді, коли пацієнт помер....",Перед навчанням моделі з датасету треба видалити всі змі...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
# ваш код: три правильні обчислення замість хибних
# Rozpodil 1
result = df['помер/жив'].value_counts(normalize=True) * 100
print(f"Вижили: {result[2]:.1f}%\nПомерли: {result[1]:.1f}%")
print()
# Rozpodil 2
clean_sex = df['Стать'].astype(str).str.strip().str.lower().str[0]
print("Розподіл статей:")
print(clean_sex.value_counts())
print()
# Rozpodil 5
spo2 = df['spO2 без дот'].copy()
spo2 = spo2.apply(lambda x: x * 100 if x < 10 else x)
print(f"Справжній мінімум сатурації: {spo2.min()}%")

Вижили: 49.6%
Померли: 50.4%

Розподіл статей:
Стать
ж    75
ч    46
Name: count, dtype: int64

Справжній мінімум сатурації: 50.0%


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    # три висновки, кожен — повне речення про ваші дані, а не загальна фраза
    "У медичному наборі даних є дуже багато порожніх клітинок у лабораторних аналізах, тому перед аналізом ці пропуски треба заповнити середніми значеннями.",
    "Через ручне введення тексту в колонках зі статтю та результатами КТ виникло багато помилок і зайвих пробілів, тому такі дані не можна аналізувати без очищення.",
    "Рахувати звичайне середнє значення або лінійну кореляцію для текстових категорій не можна, бо комп'ютер не розуміє медичної логіки і видає числа, які не мають жодного фізичного сенсу."
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. У медичному наборі даних є дуже багато порожніх клітинок у лабораторних аналізах, тому перед аналізом ці пропуски треба заповнити середніми значеннями.
2. Через ручне введення тексту в колонках зі статтю та результатами КТ виникло багато помилок і зайвих пробілів, тому такі дані не можна аналізувати без очищення.
3. Рахувати звичайне середнє значення або лінійну кореляцію для текстових категорій не можна, бо комп'ютер не розуміє медичної логіки і видає числа, які не мають жодного фізичного сенсу.


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = """Різниця між ними полягає в тому, чи існує справжній нуль. У шкалі відношень нуль означає, що чогось взагалі немає, тому такі числа можна спокійно ділити чи множити.
Натомість в інтервальній шкалі нуль обрали просто для зручності відліку.
Серед моїх даних інтервальна шкала — це дата «Виписка», а шкала відношень — результат аналізу «Фібрин2». 
Для аналізу «Фібрин2» математика працює чудово: показник 4.0 дійсно рівно вдвічі більший за 2.0. Але з датою виписки ділити не можна.
Якщо поділити 2022 рік на 2021, ми не отримаємо жодної корисної інформації, бо нульовий рік колись обрали умовно, а не тому, що до того моменту часу взагалі не існувало."""
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
У чому різниця між шкалою інтервалів і шкалою відношень? Знайдіть у файлі по одному прикладу кожної та поясніть, яка операція допустима для другої, але не для першої.

Різниця між ними полягає в тому, чи існує справжній нуль. У шкалі відношень нуль означає, що чогось взагалі немає, тому такі числа можна спокійно ділити чи множити.
Натомість в інтервальній шкалі нуль обрали просто для зручності відліку.
Серед моїх даних інтервальна шкала — це дата «Виписка», а шкала відношень — результат аналізу «Фібрин2». 
Для аналізу «Фібрин2» математика працює чудово: показник 4.0 дійсно рівно вдвічі більший за 2.0. Але з датою виписки ділити не можна.
Якщо поділити 2022 рік на 2021, ми не отримаємо жодної корисної інформації, бо нульовий рік колись обрали умовно, а не тому, що до того моменту часу взагалі не існувало.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'синтаксис',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'текст, перероблено',
    'етап 7 — висновки й відповідь':   'ні',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Я перевірив, як методи pandas обробляють різні типи шкал, переконався у правильності вибору міри зв\'язку та протестував очищення даних перед кодуванням.'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 4 готова до захисту.
Роман Бойко, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.